# Stage 2 v2 local FP32 — DenseNet-121 focal loss
Notebook chạy trực tiếp trên Windows/RTX 4050 bằng FP32 để tránh gradient overflow FP16. Mọi đường dẫn dùng `pathlib.Path`; external test luôn bị khóa.

In [1]:
from pathlib import Path
import os, sys

# Nếu Jupyter không được mở từ repository, đặt ví dụ:
# LOCAL_REPO_ROOT = Path(r"D:\Junior\ĐATH\main")
LOCAL_REPO_ROOT = None

def discover_repo_root(explicit_root=None):
    candidates = []
    if explicit_root is not None:
        candidates.append(Path(explicit_root).expanduser().resolve())
    environment_root = os.environ.get("DATH_REPO_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser().resolve())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "configs" / "default.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy repository. Hãy đặt LOCAL_REPO_ROOT bằng Path raw-string.")

REPO_DIR = discover_repo_root(LOCAL_REPO_ROOT)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

LOCAL_STAGE_ID = "stage2_focalloss_v2_local_fp32"
MODEL_NAME = "densenet121"
MODEL_SLUG = "densenet121"
BASE_CONFIG = "configs/densenet121_aug_auprc.yaml"
WEIGHTING = "sqrtcap3"
POSITIVE_WEIGHT = {"mode": "sqrt_ratio_capped", "max_value": 3.0}
RUN_FINAL_TEST = False
MAX_NEW_RUNS_PER_SESSION = None
TRAIN_BATCH_SIZE = 8
VALIDATION_BATCH_SIZE = 16
NUM_WORKERS = 0
PREFETCH_FACTOR = 2

PROCESSED_ROOT = REPO_DIR / "data" / "processed" / "consensus_v1_224"
PROTOCOL_ROOT = REPO_DIR / "data" / "protocols" / "vindr_s8515_exttest_v1"
CHECKPOINT_ROOT = REPO_DIR / "checkpoint" / LOCAL_STAGE_ID / MODEL_SLUG
LOCAL_STAGE_ROOT = REPO_DIR / "outputs" / LOCAL_STAGE_ID
RUN_LOG_ROOT = LOCAL_STAGE_ROOT / MODEL_SLUG
RUNS_ROOT = RUN_LOG_ROOT / "runs"
ANALYSIS_ROOT = RUN_LOG_ROOT / "analysis"
HARDWARE_ROOT = RUN_LOG_ROOT / "hardware"
RUNTIME_CONFIG_ROOT = LOCAL_STAGE_ROOT / "runtime_configs" / MODEL_SLUG
EXPORT_PATH = REPO_DIR / "outputs" / "exports" / "04_1_local_densenet121_stage2_focalloss_v2_local_fp32_results.zip"

assert RUN_FINAL_TEST is False
print("Repository:", REPO_DIR)
print("Processed data:", PROCESSED_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Run-log root:", RUN_LOG_ROOT)

Repository: D:\Junior\ĐATH\main
Processed data: D:\Junior\ĐATH\main\data\processed\consensus_v1_224
Checkpoint root: D:\Junior\ĐATH\main\checkpoint\stage2_focalloss_v2_local_fp32\densenet121
Run-log root: D:\Junior\ĐATH\main\outputs\stage2_focalloss_v2_local_fp32\densenet121


In [2]:
import json, platform, subprocess
import torch, torchvision
from torchvision.models import DenseNet121_Weights

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Local Stage 2 yêu cầu CUDA GPU")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_mib = int(torch.cuda.get_device_properties(0).total_memory / 1024 ** 2)
if "RTX 4050" not in gpu_name.upper():
    raise RuntimeError(f"Expected RTX 4050 Laptop GPU, got {gpu_name}")
if gpu_memory_mib < 5500:
    raise RuntimeError(f"Expected at least 5.5 GB VRAM, got {gpu_memory_mib} MiB")
if torch.__version__ != "2.7.0+cu126" or torchvision.__version__ != "0.22.0+cu126":
    raise RuntimeError(f"Use .venv-1 / requirements.txt; got torch={torch.__version__}, torchvision={torchvision.__version__}")
fp32_probe = torch.ones(8, 8, device="cuda", dtype=torch.float32) @ torch.ones(8, 8, device="cuda", dtype=torch.float32)
if not torch.isfinite(fp32_probe).all():
    raise RuntimeError("Local CUDA FP32 probe failed")
del fp32_probe
torch.cuda.empty_cache()

# Tải/kiểm tra pretrained weight trước khi run chuyển sang trạng thái running.
pretrained_state = DenseNet121_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
if not pretrained_state or not all(torch.isfinite(value).all().item() for value in pretrained_state.values() if torch.is_tensor(value)):
    raise RuntimeError("DenseNet-121 pretrained state is missing or nonfinite")
del pretrained_state

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
hardware_profile = {
    "runtime_mode": "local_windows",
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "gpu_name": gpu_name,
    "gpu_memory_mib": gpu_memory_mib,
    "nvidia_smi": smi,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "train_batch_size": TRAIN_BATCH_SIZE,
    "validation_batch_size": VALIDATION_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    "amp": False,
    "precision": "float32",
    "cudnn_benchmark": False,
    "cudnn_deterministic": True,
}
print(json.dumps(hardware_profile, indent=2))
print("Local pretrained + CUDA FP32 preflight: OK")

{
  "runtime_mode": "local_windows",
  "platform": "Windows-11-10.0.26300-SP0",
  "cpu_count": 16,
  "gpu_name": "NVIDIA GeForce RTX 4050 Laptop GPU",
  "gpu_memory_mib": 6140,
  "nvidia_smi": "NVIDIA GeForce RTX 4050 Laptop GPU, 6141, 610.62",
  "torch": "2.7.0+cu126",
  "torchvision": "0.22.0+cu126",
  "cuda": "12.6",
  "cudnn": 90701,
  "train_batch_size": 8,
  "validation_batch_size": 16,
  "num_workers": 0,
  "prefetch_factor": null,
  "amp": false,
  "precision": "float32",
  "cudnn_benchmark": false,
  "cudnn_deterministic": true
}
Local pretrained + CUDA FP32 preflight: OK


In [3]:
import copy, hashlib, yaml
from src.analysis.stage2 import analyze_stage2
from src.experiments.stage2_colab import _validate_train_data
from src.experiments.stage2_colab_v2 import (
    _run_completion, _run_subprocess_streaming, _start_telemetry, _stop_telemetry,
    build_stage2_experiments, create_or_validate_contract,
    validate_export_members, write_json_with_retry,
)
from src.utils.checksums import file_sha256
from src.utils.config import load_config

for path in (CHECKPOINT_ROOT, RUNS_ROOT, ANALYSIS_ROOT, HARDWARE_ROOT, RUNTIME_CONFIG_ROOT, EXPORT_PATH.parent):
    path.mkdir(parents=True, exist_ok=True)
data_summary = _validate_train_data(PROCESSED_ROOT, PROTOCOL_ROOT)
experiments = build_stage2_experiments(MODEL_NAME, MODEL_SLUG, BASE_CONFIG, WEIGHTING, POSITIVE_WEIGHT)
for experiment in experiments:
    experiment["run_name"] = experiment["run_name"].removesuffix("_amp") + "_fp32"
assert len(experiments) == 4 and len({item["run_name"] for item in experiments}) == 4

def build_local_runtime_config(experiment):
    config = load_config(REPO_DIR / experiment["config"])
    config["data"].update({
        "processed_root": str(PROCESSED_ROOT),
        "protocol_dir": str(PROTOCOL_ROOT),
        "train_manifest_path": str(PROTOCOL_ROOT / "train_manifest.csv"),
        "val_manifest_path": str(PROTOCOL_ROOT / "val_manifest.csv"),
    })
    config["training"].update({
        "batch_size": TRAIN_BATCH_SIZE,
        "validation_batch_size": VALIDATION_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "persistent_workers": NUM_WORKERS > 0,
        "prefetch_factor": PREFETCH_FACTOR,
        "use_amp": False,
        "show_progress": False,
        "monitor": "macro_average_precision",
        "loss": copy.deepcopy(experiment["loss"]),
    })
    runtime_dir = RUNTIME_CONFIG_ROOT / experiment["run_name"]
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_path = runtime_dir / "default.yaml"
    runtime_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return runtime_path, load_config(runtime_path)

runtime_configs = {item["run_name"]: build_local_runtime_config(item) for item in experiments}
for experiment in experiments:
    (CHECKPOINT_ROOT / experiment["run_name"]).mkdir(parents=True, exist_ok=True)
    result_dir = RUNS_ROOT / experiment["run_name"]
    result_dir.mkdir(parents=True, exist_ok=True)
    status_path = result_dir / "run_status.json"
    if not status_path.exists():
        write_json_with_retry(status_path, {"state": "pending", "run_name": experiment["run_name"]})

source_files = [
    REPO_DIR / "src/losses/classification_loss.py",
    REPO_DIR / "src/training/engine.py",
    REPO_DIR / "scripts/train.py",
    REPO_DIR / "src/experiments/stage2_colab_v2.py",
]
source_sha256 = {str(path.relative_to(REPO_DIR)): file_sha256(path) for path in source_files}
contract = {
    "stage_id": LOCAL_STAGE_ID,
    "runtime_mode": "local_windows",
    "hardware_profile": hardware_profile,
    "data_summary": data_summary,
    "source_sha256": source_sha256,
    "experiments": experiments,
    "selection_metric": "macro_average_precision",
    "external_test_used": False,
}
create_or_validate_contract(LOCAL_STAGE_ROOT / "stage2_contract.json", contract, allow_create=True)
write_json_with_retry(HARDWARE_ROOT / "hardware_profile.json", hardware_profile)
write_json_with_retry(RUN_LOG_ROOT / "experiment_registry.json", experiments)
print([item["run_name"] for item in experiments])
print("LOCAL STAGE 2 FP32 READY")

d:\Junior\ĐATH\main\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'densenet121_stage2v2_focal_g05_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'densenet121_stage2v2_focal_g10_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'densenet121_stage2v2_focal_g15_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32']
LOCAL STAGE 2 FP32 READY


In [4]:
import subprocess, traceback
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

new_runs = 0
for index, experiment in enumerate(experiments, start=1):
    config_path, config = runtime_configs[experiment["run_name"]]
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUNS_ROOT / experiment["run_name"]
    complete, resume, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
    status_path = result_dir / "run_status.json"
    if complete:
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        print(f"[{index}/4] skip complete: {experiment['run_name']}")
    else:
        if MAX_NEW_RUNS_PER_SESSION is not None and new_runs >= MAX_NEW_RUNS_PER_SESSION:
            break
        write_json_with_retry(status_path, {"state": "resuming" if resume else "running", "updated_at_utc": utc_now(), "run_name": experiment["run_name"]})
        command = [sys.executable, str(REPO_DIR / "scripts" / "train.py"), "--config", str(config_path), "--checkpoint-dir", str(checkpoint_dir), "--result-dir", str(result_dir)]
        if resume:
            command.extend(["--resume", str(checkpoint_dir / "last.pt")])
        environment = os.environ.copy()
        environment["PYTHONPATH"] = str(REPO_DIR)
        environment["PYTHONUNBUFFERED"] = "1"
        environment["PYTHONIOENCODING"] = "utf-8"
        environment["PYTHONUTF8"] = "1"
        local_subprocess_root = LOCAL_STAGE_ROOT / "subprocess_logs"
        local_log_path = local_subprocess_root / f"{result_dir.name}.log"
        telemetry_process, telemetry_handle = _start_telemetry(HARDWARE_ROOT / f"gpu_telemetry_{experiment['run_name']}.csv")
        try:
            return_code, output_tail = _run_subprocess_streaming(command, REPO_DIR, environment, result_dir, local_root=local_subprocess_root)
        except BaseException as error:
            log_tail = ""
            if local_log_path.is_file():
                log_tail = "\n".join(local_log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-200:])
            failure = {"state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "error_type": type(error).__name__, "error": str(error), "traceback": traceback.format_exc()[-12000:], "traceback_tail": log_tail, "log_file": str(local_log_path)}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise
        finally:
            try:
                _stop_telemetry(telemetry_process, telemetry_handle)
            except Exception as telemetry_error:
                print(f"Warning: telemetry cleanup failed: {telemetry_error}")
        if return_code != 0:
            failure = {"state": "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "return_code": return_code, "traceback_tail": output_tail, "log_file": "train_subprocess.log"}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise RuntimeError(f"Training failed: {experiment['run_name']}\n{output_tail}")
        complete, _, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
        if not complete:
            raise RuntimeError("Training exited without epoch-limit or early-stopping completion")
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        (result_dir / "training_failure.json").unlink(missing_ok=True)
        new_runs += 1
    partial = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
    if not partial["macro"].empty:
        display(partial["macro"])
print("Local Stage 2 training pass finished")

{'epoch': 9, 'train_loss': 0.08618323546260888, 'val_loss': 0.15374243129458692, 'val_macro_auroc': 0.9550062863128891, 'val_macro_ap': 0.4984181250632586, 'val_macro_f1_tuned': 0.5324645686619863, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.4984181250632586, 'threshold_min': 0.22504732012748718, 'threshold_mean': 0.6083942162139075, 'threshold_max': 0.9475900530815125, 'train_seconds': 434.1245065999974, 'val_seconds': 14.5584944000002, 'train_images_per_second': 29.3694546291713, 'val_images_per_second': 154.5489484132349, 'peak_gpu_memory_gb': 1.1143012046813965, 'peak_gpu_memory_allocated_gb': 1.1143012046813965, 'peak_gpu_memory_reserved_gb': 1.357421875}
{'epoch': 10, 'train_loss': 0.07756652354047287, 'val_loss': 0.16015162200232347, 'val_macro_auroc': 0.9482795505002762, 'val_macro_ap': 0.4931472089971685, 'val_macro_f1_tuned': 0.5115385358307167, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.4931472089971685, 'threshold_min': 0.052052211016416

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,None,sqrtcap3,complete,13,0.519432,0.504388,...,1304394d97a683da86c6907ba101091881496ab6a8108e...,0.504388,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


{'epoch': 1, 'train_loss': 0.15054032241363152, 'val_loss': 0.11123475411699878, 'val_macro_auroc': 0.9387432351282298, 'val_macro_ap': 0.37654902525033646, 'val_macro_f1_tuned': 0.4245337752581038, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.37654902525033646, 'threshold_min': 0.20492179691791534, 'threshold_mean': 0.42193958908319473, 'threshold_max': 0.6211613416671753, 'train_seconds': 299.67436380000436, 'val_seconds': 11.174834200006444, 'train_images_per_second': 42.54618192335281, 'val_images_per_second': 201.34526917622657, 'peak_gpu_memory_gb': 1.0891971588134766, 'peak_gpu_memory_allocated_gb': 1.0891971588134766, 'peak_gpu_memory_reserved_gb': 1.333984375}
{'epoch': 2, 'train_loss': 0.11722490581753207, 'val_loss': 0.10529422269595994, 'val_macro_auroc': 0.9491243431186289, 'val_macro_ap': 0.41378783213631687, 'val_macro_f1_tuned': 0.4510542891580314, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.41378783213631687, 'threshold_min': 0.141681

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,NaN,sqrtcap3,complete,13,0.519432,0.504388,...,1304394d97a683da86c6907ba101091881496ab6a8108e...,0.504388,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,densenet121_stage2v2_focal_g05_sqrtcap3_aug_au...,densenet121,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,8,0.523837,0.551835,...,22fdaba3a8ee91739c1d926b52fe62f9e1513555cd7b27...,0.481548,9.0,5.0,4.0,0.004405,0.047446,-0.064729,0.005649,False


{'epoch': 1, 'train_loss': 0.1086823271086403, 'val_loss': 0.08011516600681676, 'val_macro_auroc': 0.9415697602314399, 'val_macro_ap': 0.3809054074070727, 'val_macro_f1_tuned': 0.41895409775351367, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.3809054074070727, 'threshold_min': 0.24893724918365479, 'threshold_mean': 0.4372765954051699, 'threshold_max': 0.5935190320014954, 'train_seconds': 271.95478609998827, 'val_seconds': 10.5257055000111, 'train_images_per_second': 46.88279321295809, 'val_images_per_second': 213.76239340893846, 'peak_gpu_memory_gb': 1.0891971588134766, 'peak_gpu_memory_allocated_gb': 1.0891971588134766, 'peak_gpu_memory_reserved_gb': 1.333984375}
{'epoch': 2, 'train_loss': 0.08557819881202544, 'val_loss': 0.08029548818369707, 'val_macro_auroc': 0.9450730720027722, 'val_macro_ap': 0.40607907512711394, 'val_macro_f1_tuned': 0.4479866089487309, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.40607907512711394, 'threshold_min': 0.27686256170

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,NaN,sqrtcap3,complete,13,0.519432,0.504388,...,1304394d97a683da86c6907ba101091881496ab6a8108e...,0.504388,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,densenet121_stage2v2_focal_g05_sqrtcap3_aug_au...,densenet121,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,8,0.523837,0.551835,...,22fdaba3a8ee91739c1d926b52fe62f9e1513555cd7b27...,0.481548,9.0,5.0,4.0,0.004405,0.047446,-0.064729,0.005649,False
2,densenet121_stage2v2_focal_g10_sqrtcap3_aug_au...,densenet121,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,9,0.528095,0.546459,...,d885ef8a9ff69a80ac8dcb2f45457bdffe2bf0e2c78a89...,0.477409,9.0,5.0,4.0,0.008663,0.042070,-0.025098,0.021241,False


{'epoch': 1, 'train_loss': 0.0785986944349373, 'val_loss': 0.05717293066117499, 'val_macro_auroc': 0.9435737128951615, 'val_macro_ap': 0.38511031585471284, 'val_macro_f1_tuned': 0.4310548451690071, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.38511031585471284, 'threshold_min': 0.3246489465236664, 'threshold_mean': 0.4525384796517236, 'threshold_max': 0.6004747152328491, 'train_seconds': 519.2221810999908, 'val_seconds': 18.343111599999247, 'train_images_per_second': 24.555961713709276, 'val_images_per_second': 122.6618498030668, 'peak_gpu_memory_gb': 1.0891971588134766, 'peak_gpu_memory_allocated_gb': 1.0891971588134766, 'peak_gpu_memory_reserved_gb': 1.333984375}
{'epoch': 2, 'train_loss': 0.061623909178025585, 'val_loss': 0.05841390193998813, 'val_macro_auroc': 0.9474107478062556, 'val_macro_ap': 0.4251824457601046, 'val_macro_f1_tuned': 0.4730199863295083, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.4251824457601046, 'threshold_min': 0.40298590064

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,NaN,sqrtcap3,complete,13,0.519432,0.504388,...,1304394d97a683da86c6907ba101091881496ab6a8108e...,0.504388,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,densenet121_stage2v2_focal_g05_sqrtcap3_aug_au...,densenet121,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,8,0.523837,0.551835,...,22fdaba3a8ee91739c1d926b52fe62f9e1513555cd7b27...,0.481548,9.0,5.0,4.0,0.004405,0.047446,-0.064729,0.005649,False
2,densenet121_stage2v2_focal_g10_sqrtcap3_aug_au...,densenet121,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,9,0.528095,0.546459,...,d885ef8a9ff69a80ac8dcb2f45457bdffe2bf0e2c78a89...,0.477409,9.0,5.0,4.0,0.008663,0.042070,-0.025098,0.021241,False
3,densenet121_stage2v2_focal_g15_sqrtcap3_aug_au...,densenet121,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,10,0.525733,0.552978,...,2f02c72a499f1d9d41325d64e685bac09b02793da16bf4...,0.481582,10.0,4.0,3.0,0.006302,0.048590,-0.014136,0.017114,False


Local Stage 2 training pass finished


In [5]:
import shutil, zipfile
analysis = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
display(analysis["macro"])
display(analysis["ranking"])
manifest = {"created_at_utc": utc_now(), "stage_id": LOCAL_STAGE_ID, "runtime_mode": "local_windows", "model": MODEL_NAME, "checkpoint_root": str(CHECKPOINT_ROOT), "run_log_root": str(RUN_LOG_ROOT), "selection_metric": "macro_average_precision", "external_test_used": False}
write_json_with_retry(RUN_LOG_ROOT / "experiment_manifest.json", manifest)
archive_base = EXPORT_PATH.with_suffix("")
if EXPORT_PATH.exists():
    EXPORT_PATH.unlink()
shutil.make_archive(str(archive_base), "zip", root_dir=LOCAL_STAGE_ROOT)
with zipfile.ZipFile(EXPORT_PATH) as archive:
    validate_export_members(archive.namelist())
print("Export ready:", EXPORT_PATH)
print("External test locked.")

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,NaN,sqrtcap3,complete,13,0.519432,0.504388,...,1304394d97a683da86c6907ba101091881496ab6a8108e...,0.504388,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,densenet121_stage2v2_focal_g05_sqrtcap3_aug_au...,densenet121,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,8,0.523837,0.551835,...,22fdaba3a8ee91739c1d926b52fe62f9e1513555cd7b27...,0.481548,9.0,5.0,4.0,0.004405,0.047446,-0.064729,0.005649,False
2,densenet121_stage2v2_focal_g10_sqrtcap3_aug_au...,densenet121,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,9,0.528095,0.546459,...,d885ef8a9ff69a80ac8dcb2f45457bdffe2bf0e2c78a89...,0.477409,9.0,5.0,4.0,0.008663,0.042070,-0.025098,0.021241,False
3,densenet121_stage2v2_focal_g15_sqrtcap3_aug_au...,densenet121,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,10,0.525733,0.552978,...,2f02c72a499f1d9d41325d64e685bac09b02793da16bf4...,0.481582,10.0,4.0,3.0,0.006302,0.048590,-0.014136,0.017114,False


,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible,variant_order,rank
0,densenet121_stage2v2_bce_sqrtcap3_aug_auprc_s8...,densenet121,bce,weighted_bce,NaN,sqrtcap3,complete,13,0.519432,0.504388,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True,0,1
1,densenet121_stage2v2_focal_g05_sqrtcap3_aug_au...,densenet121,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,8,0.523837,0.551835,...,9.0,5.0,4.0,0.004405,0.047446,-0.064729,0.005649,False,1,2
2,densenet121_stage2v2_focal_g10_sqrtcap3_aug_au...,densenet121,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,9,0.528095,0.546459,...,9.0,5.0,4.0,0.008663,0.042070,-0.025098,0.021241,False,2,3
3,densenet121_stage2v2_focal_g15_sqrtcap3_aug_au...,densenet121,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,10,0.525733,0.552978,...,10.0,4.0,3.0,0.006302,0.048590,-0.014136,0.017114,False,3,4


Export ready: D:\Junior\ĐATH\main\outputs\exports\04_1_local_densenet121_stage2_focalloss_v2_local_fp32_results.zip
External test locked.
